# Notebook 06: Physiological Signals Analysis
## 3D Facial Landmarks, Eye Aspect Ratio (EAR), and Mouth Aspect Ratio (MAR)

### Phase 2B — Physiological Signals & Temporal Analysis
This notebook implements an experimental, notebook-first investigation of facial landmark extraction, Eye Aspect Ratio (EAR), and Mouth Aspect Ratio (MAR).

**Key Objectives:**
1. Environment and dependency verification.
2. Frame ingestion from `test_video.mp4`.
3. 3D Facial landmark mesh extraction (MediaPipe 478 points).
4. Eye landmark geometry and EAR formulation (Soukupová & Čech).
5. Mouth landmark geometry and MAR formulation.
6. Temporal extraction over the entire stream.
7. Threshold sensitivity exploration (EAR: 0.18–0.25; MAR: 0.40–0.70).
8. Comparative analysis: YOLO eye/yawn detections vs. physiological metrics.
9. Failure case and occlusion analysis.



In [ ]:
# 1. Environment Verification & Imports
import sys
import time
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np

# Ensure project root is accessible
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import AppConfig
from src.detector import YOLODetector
from src.physiological import FaceLandmarkDetector, EARCalculator, MARCalculator

print("Environment verified successfully.")
print(f"OpenCV Version: {cv2.__version__}")
print(f"NumPy Version: {np.__version__}")



### 2. Video Loading & Sample Frame Extraction
We load the baseline evaluation asset `test_video.mp4` and sample frames representing:
- Neutral open-eyed state
- Eye-closure state
- Yawn / wide mouth opening state



In [ ]:
# Load video source and extract metadata
video_path = PROJECT_ROOT / "test_video.mp4"
cap = cv2.VideoCapture(str(video_path))
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"Video Asset: {video_path.name}")
print(f"Resolution: {width}x{height} @ {fps:.2f} FPS | Total Frames: {total_frames}")

# Extract sample frames for landmark inspection
sample_frames = {}
target_indices = [10, 150, 270, 480]

for idx in range(total_frames):
    ret, frame = cap.read()
    if not ret:
        break
    if idx in target_indices:
        sample_frames[idx] = frame.copy()
    if len(sample_frames) == len(target_indices):
        break
cap.release()
print(f"Extracted {len(sample_frames)} sample frames at indices {list(sample_frames.keys())}")



### 3. 3D Facial Landmark Detection
We employ the MediaPipe 3D FaceLandmarker pipeline (478 3D points), which provides normalized coordinates $[0, 1]$ mapped to image pixel coordinates $(x, y, z)$.



In [ ]:
# Initialize Face Landmark Detector
landmark_detector = FaceLandmarkDetector(model_path=str(PROJECT_ROOT / "face_landmarker.task"))
ear_calc = EARCalculator(threshold=0.21)
mar_calc = MARCalculator(threshold=0.55)

test_frame = sample_frames[10]
res = landmark_detector.detect(test_frame)

print(f"Face Detected: {res.detected}")
print(f"Landmark Array Shape: {res.pixel_landmarks.shape if res.pixel_landmarks is not None else None}")
print(f"Bounding Box: {res.bbox}")



### 4. Landmark Visualization & Overlay
We visualize the 478 3D facial landmarks and specifically highlight the ocular and labial feature points.



In [ ]:
# Visualize Facial Landmarks on Sample Frame
vis_frame = test_frame.copy()
if res.detected and res.pixel_landmarks is not None:
    # Draw face mesh points (subtle)
    for pt in res.pixel_landmarks[::3]:  # Subsample for clarity
        cv2.circle(vis_frame, (int(pt[0]), int(pt[1])), 1, (0, 255, 0), -1)

    # Highlight Left Eye Points (Indices: 33, 160, 158, 133, 153, 144)
    left_indices = [33, 160, 158, 133, 153, 144]
    for idx in left_indices:
        pt = res.pixel_landmarks[idx]
        cv2.circle(vis_frame, (int(pt[0]), int(pt[1])), 3, (255, 0, 0), -1)

    # Highlight Right Eye Points (Indices: 362, 385, 387, 263, 373, 380)
    right_indices = [362, 385, 387, 263, 373, 380]
    for idx in right_indices:
        pt = res.pixel_landmarks[idx]
        cv2.circle(vis_frame, (int(pt[0]), int(pt[1])), 3, (0, 0, 255), -1)

    # Highlight Mouth Inner Lip Points
    mouth_indices = [61, 291, 13, 14, 78, 308]
    for idx in mouth_indices:
        pt = res.pixel_landmarks[idx]
        cv2.circle(vis_frame, (int(pt[0]), int(pt[1])), 3, (0, 255, 255), -1)

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(vis_frame, cv2.COLOR_BGR2RGB))
plt.title("MediaPipe 3D Landmarks: Eye (Blue/Red) and Mouth (Yellow) Geometry")
plt.axis("off")
plt.show()



### 5. Eye Landmark Geometry & EAR Formulation
Eye Aspect Ratio (Soukupová and Čech, 2016) calculates the ratio of eye opening height to horizontal width:
$$\text{EAR} = \frac{\|p_2 - p_6\| + \|p_3 - p_5\|}{2 \cdot \|p_1 - p_4\|}$$
where:
- $p_1, p_4$ are the lateral and medial canthi (eye corners).
- $p_2, p_6$ and $p_3, p_5$ are vertical upper and lower eyelid landmark pairs.



In [ ]:
# 6 & 7 & 8: Calculate Left, Right, and Average EAR across Sample Frames
for frame_idx, frame in sample_frames.items():
    res = landmark_detector.detect(frame)
    if res.detected:
        ear_res = ear_calc.calculate(res.pixel_landmarks)
        print(f"Frame {frame_idx:03d} -> Left EAR: {ear_res.left_ear:.3f} | Right EAR: {ear_res.right_ear:.3f} | Avg EAR: {ear_res.average_ear:.3f} | Closed: {ear_res.is_closed}")



### 9. EAR & MAR Stream Analysis
We load the synchronized telemetry extracted across all 617 frames of `test_video.mp4` to visualize continuous EAR and MAR dynamics.



In [ ]:
# Load pre-extracted continuous telemetry
import csv

telemetry_path = PROJECT_ROOT / "results" / "phase2b" / "telemetry.csv"
timestamps = []
ears = []
mars = []
closed_flags = []

with open(telemetry_path, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        timestamps.append(float(row["timestamp"]))
        ears.append(float(row["average_ear"]) if row["average_ear"] else np.nan)
        mars.append(float(row["mar"]) if row["mar"] else np.nan)
        closed_flags.append(row["eye_closed"].lower() == "true")

print(f"Loaded {len(timestamps)} frames of telemetry data.")

# Plot Continuous EAR and MAR
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax1.plot(timestamps, ears, color="#1f77b4", label="Average EAR", lw=1.2)
ax1.axhline(0.21, color="red", linestyle="--", label="EAR Threshold (0.21)")
ax1.set_ylabel("EAR")
ax1.set_title("Stream Trajectory: Eye Aspect Ratio (EAR) & Mouth Aspect Ratio (MAR)")
ax1.grid(True, alpha=0.3)
ax1.legend(loc="upper right")

ax2.plot(timestamps, mars, color="#9467bd", label="MAR", lw=1.2)
ax2.axhline(0.55, color="red", linestyle="--", label="MAR Yawn Threshold (0.55)")
ax2.set_xlabel("Time (seconds)")
ax2.set_ylabel("MAR")
ax2.grid(True, alpha=0.3)
ax2.legend(loc="upper right")
plt.tight_layout()
plt.show()



### 10. EAR Distribution & Threshold Calibration
We plot the frequency distribution of EAR values to analyze the bimodal distribution between open and closed eye states.



In [ ]:
# Histogram of EAR Values
valid_ears = [e for e in ears if not np.isnan(e)]

plt.figure(figsize=(9, 4))
plt.hist(valid_ears, bins=35, color="steelblue", edgecolor="black", alpha=0.7)
plt.axvline(0.18, color="green", linestyle=":", lw=2, label="Conservative (0.18)")
plt.axvline(0.21, color="red", linestyle="--", lw=2, label="Calibrated (0.21)")
plt.axvline(0.24, color="orange", linestyle=":", lw=2, label="Sensitive (0.24)")
plt.title("Distribution of EAR Values (617 Frames)")
plt.xlabel("Eye Aspect Ratio")
plt.ylabel("Frame Count")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



### 11 & 12. Mouth Aspect Ratio (MAR) Formulation & Yawn Dynamics
$$\text{MAR} = \frac{\|p_{\text{upper}} - p_{\text{lower}}\|}{\|p_{\text{corner\_left}} - p_{\text{corner\_right}}\|}$$
A threshold of $0.55$ reliably isolates wide mouth openings associated with yawning while ignoring normal mouth movements during passive posture.



In [ ]:
# MAR Distribution and Thresholding
valid_mars = [m for m in mars if not np.isnan(m)]

plt.figure(figsize=(9, 4))
plt.hist(valid_mars, bins=35, color="salmon", edgecolor="black", alpha=0.7)
plt.axvline(0.55, color="red", linestyle="--", lw=2, label="Yawn Threshold (0.55)")
plt.title("Distribution of MAR Values")
plt.xlabel("Mouth Aspect Ratio")
plt.ylabel("Frame Count")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



### 13 & 14. YOLO vs Physiological Metric Comparison
We cross-reference YOLO bounding-box classifications with geometric EAR and MAR values.

**Observations:**
- YOLO provides bounding-box detections with class probabilities but fluctuates with subtle squinting.
- EAR provides a continuous, analog metric that captures partial closure gradients and blink velocity.
- MAR accurately distinguishes wide yawns from speech without requiring retraining.



In [ ]:
# Summary Statistics
mean_ear = np.nanmean(ears)
std_ear = np.nanstd(ears)
mean_mar = np.nanmean(mars)
closed_pct = (sum(closed_flags) / len(closed_flags)) * 100.0

print("=== PHYSIOLOGICAL SIGNAL SUMMARY ===")
print(f"Mean EAR: {mean_ear:.3f} +/- {std_ear:.3f}")
print(f"Mean MAR: {mean_mar:.3f}")
print(f"Proportion of Closed-Eye Frames (EAR < 0.21): {closed_pct:.1f}%")
print(f"Phase 2B Calibrated EAR Threshold: 0.21")
print(f"Phase 2B Calibrated MAR Threshold: 0.55")



### 15. Failure Mode & Edge-Case Analysis
1. **Severe Head Rotation**: When the driver turns $>45^\circ$, the contralateral eye is occluded, causing EAR to drop or fail.
2. **Partial Face**: When the chin or forehead leaves the frame, landmark tracking can drift.
3. **Eyeglasses / Reflection**: Strong glare on spectacle lenses can distort eyelid landmark localization.
4. **Lighting Conditions**: Under extreme low light, contrast degrades.



### 16. Conclusion
The combination of 3D facial landmarks with EAR (0.21) and MAR (0.55) provides reliable, sub-frame physiological tracking.
In Notebook 07, these instantaneous metrics are processed into time-domain signals (Blinks, Yawn Duration, and Rolling PERCLOS).

